# 42 — Surface-target screen, **skin**: cohort fix, artifact tests, malignant vs reactive CD4

The skin twin of 43, rebuilt on the same template. The first pass ranked **ENO1, ANXA2, HSP90AA1,
VDAC1, GPI, CALU and QSOX1** at the top of the prevalence axis and put **COL6A3, LGALS3 and GRN**
on the shortlist, while the antigens whose answer is already known came out at **CCR4 rank 386,
KIR3DL2 533 and IL2RA 1,968 of 2,115 with a negative log2FC**. Only TNFRSF8 (CD30) was anywhere
near the top. A screen that cannot recover the drugs that already work in this disease is not yet
measuring selectivity, and its novel hits are not evidence.

Two things are wrong with that pass, and they are not the same thing.

**1. The cohort, which was wrong and is now fixed.** 146 donor keys are not 146 patients. Nineteen
carry a malignant clone that is **not CD4** — CD8, γδ or double-negative by `clin_phenotype`, plus
the CD8 aggressive cytotoxic case and four pcAECTCL — so `mal_cluster` finds no tumour in their CD4
compartment and all of their CD4 cells sit in the **reactive** arm of a tumour-vs-reactive test.
Fifteen more carry a diagnosis outside the MF/SS spectrum (LyP, SPP, LPP). Ten patients were
biopsied twice or more and appear under up to four keys each. §0b removes them: **109 independent
donor keys, 107 people**, diagnosis from `clin_dx`.

**2. The ranking's shape, which is what §2 is for.** ER, mitochondrial and glycolytic transcripts
at the top of a *surface* screen, stromal collagen on the shortlist, and `frac_donors_up` near 0.9
for all of them is the signature of something that scales with the cell rather than with the
antigen. Skin offers two mechanisms, and they need different controls:

- **stromal and epidermal contamination.** Dissociating skin shears keratinocytes, and keratin,
  collagen and melanocyte transcripts are the classic skin ambient and doublet load. A malignant
  cluster that is denser or dirtier picks up more of it.
- **a biosynthetic-activity axis.** A larger, more transcriptionally active malignant cell raises
  every abundant housekeeping transcript at once, including the secretory machinery, without any
  gene being selectively expressed. Depth matching, trimming and the per-cell UMI coupling are the
  tests for this one; the contamination null cannot see it.

§2 puts six tests to both, against an **empirical null** made of transcripts a CD4 T cell cannot
make — the keratins, the collagens, VWF, PMEL, the tryptases, LYZ, HBB. Those genes are what
contamination scores in this cohort, so the thresholds are read off them rather than chosen, and
the section is allowed to conclude that **nothing is excluded**: the calibration verdict prints
before any flag is assigned.

| § | what it does |
|---|---|
| §0b | the cohort fix: 146 donor keys → the independent set, diagnosis from `clin_dx` |
| §1 | input, on the rebuilt QC tables |
| **§2** | **the artifact tests** — six against an empirical contamination null, plus the TCR-call control |
| §3 | malignant vs reactive CD4 on the surviving genes, with per-donor figures |
| §4 | on-target off-tumour: the other T arms, the non-T skin compartments, healthy skin |
| §5–§6 | surface plausibility and escape risk |
| §7 | the ranked table, the control check, and the surfaceome plausibility layer |
| §8 | dossier |
| §9 | the numerical hand-off block |

**Inputs.** `jobs/run_target_screen_qc.sh --tissue skin`, which supersedes `run_target_screen.sh`
for this compartment too: true-library CP10K, robust location statistics, a contamination-filtered
recompute, depth matching, per-cell couplings to the keratinocyte / fibroblast / myeloid modules,
and the lineage-control genes. §2d additionally needs the same job with
`--call mal_tcr --no-cluster --out-dir data/targets/by_call_mal_tcr`. §4 also reads the degradome
pseudobulk cube for the non-T compartments and Reynolds 2021 for healthy skin.

**Outputs** — `tables/tgt42_*`, `figures/targets/tgt42_*`.

> Unlike blood, the skin object carries **all 42,347 genes**, so `CD4` and `TNFRSF8` are present
> and all four positive controls are testable. A gene missing from the screen here is missing
> because the panel ∪ surfaceome selection did not ask for it, not because an HVG filter dropped
> it.

> **This notebook cannot call a target.** It ranks transcripts. CTLA4 and PDCD1 are internalised,
> SELL is shed by ADAM17, and nothing here measures epitope density. There is no CITE-seq in skin,
> so the protein check that 45 runs for blood has no skin equivalent: every candidate below needs
> flow or imaging before anyone says the word target.

In [ ]:
# ============================================================================
# §0  Setup
# ============================================================================
import warnings; warnings.filterwarnings("ignore")
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


def _resolve_nb_dir() -> Path:
    start = Path.cwd()
    for base in [start, *start.parents]:
        for sub in [Path("."), Path("MF")]:
            cand = base / sub
            if cand.name == "MF" and (cand / "data").exists():
                return cand.resolve()
    raise FileNotFoundError(f"could not locate MF/data from {start}")


NB_DIR = _resolve_nb_dir()
sys.path.insert(0, str(NB_DIR / "helpers"))

import surface_targets as ST
import target_screen as TS
import target_cohort as TC
import target_qc as TQ
import target_figs as TF
import target_het as TH
import cluster_malig_helpers as CM
import malignancy as MAL
import degradome_data as dcfg          # the non-T pseudobulk cube lives in the degradome build
import external_ref as X               # Reynolds 2021 healthy skin

# ALL THREE, before any path is read. `malignancy` and `target_qc` both default to one
# compartment at import -- `malignancy` to skin, `target_qc` to blood -- so a notebook that sets
# only one of them gets the other's parquets or the other's contaminants, and the first symptom
# is an unrelated KeyError several cells later. The asserts below are the guard, not the comment.
CM.use_tissue("skin")
MAL.use_tissue("skin")
TQ.use_tissue("skin")
assert MAL.CLUSTER_PQ.name.startswith("skin_"), MAL.CLUSTER_PQ
assert "KRT14" in TQ.BAND_GENES, TQ.BAND_GENES

TISSUE = "skin"
TARGETS = NB_DIR / "data" / "targets"
PREFIX = "tgt42_"
MALIG, REACT, HC_CD4 = "CD4_malignant", "CD4_reactive", "CD4_HC"
TREG, CD8, UNASSESSED = "CD4_Treg", "CD8", "CD4_unassessed"

# Gates. Stated once, here, so every table below can be read against them.
MIN_CELLS = 20        # a donor's arm must have this many cells to contribute a point
MIN_DONORS = 5        # a gene needs this many paired donors to be tested
DETECT_FLOOR = 0.10   # "the antigen is present" for prevalence purposes

ST.TAB_PREFIX = ST.FIG_PREFIX = PREFIX
TF.rc()
pd.set_option("display.width", 240)

print("NB_DIR =", NB_DIR)
print("tissue:", CM.TISSUE, "| mal_cluster source:", MAL.CLUSTER_PQ.name,
      "| headline:", MAL.HEADLINE)
print("contamination null:", len(TQ.BAND_GENES), "genes |",
      "coupling modules:", TQ.COUPLING_MODULES[TISSUE])

## §0b — The cohort, before any statistic

`donor` in the target tables is not a patient. Three corrections, all read from the curated
`clin_*` columns (`docs/CLINICAL_METADATA.md`), none of them guessed here:

- **the clone is not CD4.** `clin_phenotype` is the lineage of the *malignant* clone, and 19 keys
  carry CD8, γδ or double-negative disease — the four pcAECTCL cases, D6 MF5 (CD8 aggressive
  cytotoxic, `CLINICAL_METADATA.md:145`), the Berti CD8 and γδ-MF patients of D1, and two DN. The
  malignancy call looks for a malignant CD4 cluster and finds none, so **every one of their CD4
  cells enters `CD4_reactive`** — the benign side of the comparison — while their actual tumour
  sits in the CD8 arm. This is the correction that matters most and it has nothing to do with
  pseudo-replication.
- **the diagnosis is outside the MF/SS spectrum.** Seven LyP, six SPP and two LPP keys. LyP is a
  CD30+ lymphoproliferative disorder and parapsoriasis is a precursor state; both are interesting
  and neither belongs inside an MF/SS tumour-vs-reactive contrast.
- **one patient under several keys.** Ten patients were biopsied more than once —
  `vienna:MF318` under four keys, `vienna:MF311` under three, `pittsburgh:MF19` in both this
  atlas and the blood one. The lesional sample is kept (the deepest, among lesions), the rest are
  dropped. Keys that differ only by an HTO index are **not** collapsed: an HTO index demultiplexes
  different donors that share a patient key, which is the `D13__HB1-3` case in blood.

Diagnosis comes from `clin_dx` throughout. The atlas `disease` column disagrees with it for 27
keys — it files every LyP, SPP, LPP and pcAECTCL case under `CTCL_other` and calls six SS patients
MF — and is not read again in this notebook.

Unlike blood, no healthy donor is mislabelled here: `disease == "HC"` and `clin_dx == "HC"` name
the same 22 keys, so the `CD4_HC` arm needed no repair.

`burden` — the malignant share of a donor's CD4 compartment — is attached here. §2 uses it as the
mechanistic test for contamination.

In [ ]:
# ============================================================================
# §0b  Cohort: which donor keys are independent MF/SS patients with a CD4 clone
# ============================================================================
don = TC.skin_donor_table()
don.to_csv(ST.tab("skin_cohort"), index=True)

_raw = pd.read_parquet(TARGETS / f"{TISSUE}_target_long.parquet")
long = TC.filter_long(_raw, don)
bur = TC.burden(long)
don = don.join(bur, how="left")

print(TC.cohort_report(don, long))
print("\nwrote", ST.tab("skin_cohort"))

print("\n" + "=" * 78)
print("dropped keys, every one with its reason:")
print(don.loc[~don["cohort_keep"], ["study", "dx", "phenotype", "lesion", "person", "n_cells",
                                    "drop_reason"]]
      .sort_values(["drop_reason", "study"]).to_string())

print("\n" + "=" * 78)
print("the independent cohort, by what it can be split on:")
_k = don[don["cohort_keep"]]
for col, label in (("dx", "diagnosis"), ("lesion", "lesion biopsied"),
                   ("clin_stage_class", "stage class"), ("sex", "sex")):
    print(f"  {label:18s} " + ", ".join(f"{k}:{v}" for k, v in
                                        _k[col].value_counts(dropna=False).items()))
print(f"  {'age':18s} {int(_k['age'].notna().sum())} donors, median "
      f"{_k['age'].median():.0f}")
print(f"  {'treatment known':18s} {int(_k['tx_known'].sum())} donors")
print("\nburden -- the malignant share of a donor's CD4 compartment, from the arm cell counts "
      "themselves. In skin it is a biopsy property as much as a patient property (a tumour "
      "lesion is not a patch), and §2 uses it as the mechanistic test for contamination:")
print(_k["burden"].describe(percentiles=[.25, .5, .75]).round(3).to_string())

## §1 — Input, and the four things that make skin different from blood

1. **The object carries all 42,347 genes.** The screen tests the panel ∪ surfaceome selection
   (~2,100 genes), but absence from that selection is a choice this project made, not an HVG
   filter: `CD4` and `TNFRSF8` are both present, so all four positive controls are testable —
   which is why §7's control check is a sharper instrument here than in blood.
2. **The malignant fraction is a property of the lesion.** Blood in SS is ~92% malignant CD4 and
   the reactive arm barely exists; a skin biopsy ranges from a patch with a small clone to a
   tumour that is almost all clone. 94 of 97 donors with a malignant arm also have a reactive
   one, so the paired test is well powered — and `burden` spans most of its range, which makes
   the burden-scaling test in §2 far more informative than it can be in blood.
3. **The contaminants are the tissue.** Keratinocytes, fibroblasts, endothelium, melanocytes,
   mast cells and macrophages are all present in the same dissociation, and keratin and collagen
   are the classic skin ambient transcripts. `CD4_unassessed` (14 donors, 39,692 cells) is carried
   but never used as a comparator: those are cells the call could not be evaluated on, and
   folding them into "reactive" would make the reactive arm mean "malignant was not detected".
4. **The tables are the QC build.** `mean_all` is log1p CP10K on the **true** library
   (`obs.total_counts`), with `trim1_mean`, `wins90_mean`, `mean_all_clean` (lineage-positive and
   high-doublet cells dropped) and `mean_all_ds` (binomially thinned to a common depth) beside
   it. Those four columns are what separates a selective antigen from a cell that is simply
   making more of everything.

In [ ]:
# ============================================================================
# §1  Load the rest of the QC build
# ============================================================================
meta = TC.filter_long(pd.read_parquet(TARGETS / f"{TISSUE}_target_meta.parquet"), don)
man = json.loads((TARGETS / f"{TISSUE}_target_manifest.json").read_text())
cl_path = TARGETS / f"{TISSUE}_target_cluster.parquet"
cluster_long = TC.filter_long(pd.read_parquet(cl_path), don) if cl_path.exists() else None
qc_path = TARGETS / f"{TISSUE}_target_qc_cell.parquet"
qc_cell = pd.read_parquet(qc_path) if qc_path.exists() else None

assert man.get("builder") == "run_target_screen_qc.py", (
    "these tables came from the v1 builder, which has no robust statistics, no clean recompute "
    "and no lineage controls -- run jobs/run_target_screen_qc.sh --tissue skin first. "
    f"manifest says: {man.get('builder', 'run_target_screen.py')}")

print(f"source {Path(man['source']).name} | call {man['malignancy_call']} | "
      f"{man['n_genes_in_object']:,} genes in object -> "
      f"{man['gene_selection'].get('n_total', man['gene_selection'].get('n_screened')):,} "
      f"screened (+{man['gene_selection'].get('n_lineage_added', 0)} lineage controls, "
      f"+{man['gene_selection'].get('n_traffic_added', 0)} trafficking)")
print("library:", man.get("library"))
print("clean rule:", json.dumps(man.get("clean_rule", {}), default=str))
print(f"depth matching: {man.get('ds_depth'):,} UMI on {man.get('frac_cells_ds', 0):.1%} "
      "of cells")
print("coupling modules:", man.get("coupling_modules"))

arms = (meta.groupby("arm").agg(donors=("donor", "nunique"), cells=("n_cells", "sum"),
                                studies=("study", "nunique"))
        .sort_values("cells", ascending=False))
print("\narms, independent cohort:")
print(arms.to_string())

paired = meta[meta["arm"].isin([MALIG, REACT])].groupby("donor")["arm"].nunique().eq(2)
PAIRED_DONORS = sorted(paired.index[paired])
print(f"\ndonors with both arms: {len(PAIRED_DONORS)} of "
      f"{meta.loc[meta['arm'].eq(MALIG), 'donor'].nunique()} with a malignant arm")
DONOR_STUDY = dict(zip(meta["donor"], meta["study"]))

print("\ncontamination load per arm (median over donors) -- the asymmetry an artifact needs:")
_cl = [c for c in ("frac_lineage_pos", "median_total_counts", "median_doublet_score",
                   "score_keratinocyte", "score_fibroblast", "score_myeloid",
                   "score_endothelial", "score_melanocyte", "score_mast")
       if c in meta.columns]
print(meta.groupby("arm")[_cl].median().round(4).to_string())
print("\nA malignant arm that is dirtier than its reactive arm on these rows is the precondition "
      "for the whole of §2. One that is not says the contamination story cannot be the "
      "explanation, whatever the gene names look like.")

## §2 — Is the ranking an artifact? A test that is allowed to fail

The charge against the first ranking is specific, so the test can be too.

**Two mechanisms, not one.**

*Contamination.* A skin dissociation shears keratinocytes and releases collagen; those transcripts
enter droplets as ambient RNA and as doublets. Ambient RNA is shared by both arms and cannot by
itself make a gene look selective — but doublets are not shared: if the malignant clusters are
denser, dirtier or sit in a different part of the lesion, they take up more of the tissue around
them, and every stromal transcript rises in the malignant arm in every patient at once.

*Biosynthetic activity.* A malignant T cell that is larger and more transcriptionally active than
the small reactive lymphocyte beside it raises **every** abundant transcript — ENO1, HSP90AA1,
VDAC1, CALU, the whole ER and glycolytic programme — without expressing any of them selectively.
This produces exactly the first pass's top list, and the contamination null cannot see it: it is
not contamination. The tests that can are depth matching (`mean_all_ds`), trimming, and the
per-cell coupling to library size.

**Six tests**, each with its own failure mode:

| test | what fires it |
|---|---|
| `fails_trim` | the effect collapses when the top 1% of cells is trimmed — carried by a handful of cells, not by the population |
| `fails_concentration` | more of the arm's counts sit in its top 1% of cells than for other genes *detected at the same rate* |
| `fails_lineage` | per-cell expression tracks the keratinocyte, fibroblast or myeloid module score **inside** the malignant arm |
| `fails_umi` | it tracks per-cell library size, or detection is confined to the top-UMI decile, beyond what its prevalence implies |
| `fails_clean` | it does not survive dropping lineage-positive and high-doublet-score cells |
| `fails_burden` | its per-donor effect grows with that donor's malignant:reactive abundance ratio |

Two failures ⇒ `artifact`, one ⇒ `suspect`. Only up-in-malignant genes can be flagged;
contamination entering the malignant arm cannot push a gene down.

**The null, and the escape hatch.** The keratins, `KRTDAP`, `DMKN`, `SFN`, the collagens, `DCN`,
`LUM`, `VWF`, `CDH5`, `CLDN5`, `PMEL`, `MLANA`, `TYRP1`, `DCT`, `LYZ`, `CD14`, `CD68`, `C1QA/B`,
`CD163`, the tryptases, `CPA3`, `MS4A1`, `CD79A`, `JCHAIN`, `MZB1`, `HBB` and `HBA2` are not CD4
T-cell transcripts, so whatever they score *is* what contamination scores in this cohort, and
every threshold is read off them rather than chosen. That also makes the section refutable: **if
those genes are not up in the malignant arm, the mechanism is not operating here, and §2b excludes
nothing.** The calibration verdict prints before any flag is assigned — read it first, because it
decides whether the rest of the section means anything. `PECAM1` (CD31) and `HLA-DRA` are
deliberately out of the null: T cells carry both.

**Why the statistics are prevalence-matched.** `top1_share`, the top-UMI-decile odds ratio and the
module correlations are all functions of how often a gene is detected at all: a gene seen in two
cells necessarily concentrates, and a gene seen in 80% of cells necessarily rises with depth
because detection saturates. Compared raw, the band would condemn CD7 and acquit a gene seen once.
Each is taken as a residual against genes of the same detection rate.

**§2d adds the two artifacts that are not contamination** — the malignancy call being a cluster
label while the comparison is between those same clusters, and the effect living in one or two
clusters rather than in the clone. The control for the first is the whole screen rebuilt with the
arms taken from the TCR clone, which is measured independently of the transcriptome.

In [ ]:
# ============================================================================
# §2a  The six artifact statistics, and the empirical contamination null
# ============================================================================
# The same paired test, five times, on five location estimators. An effect carried by a handful
# of cells survives only the first; an effect that is really a depth difference dies in the last.
sel = TS.paired_selectivity(long, MALIG, REACT, value="mean_all", min_donors=MIN_DONORS)
sel_trim = TS.paired_selectivity(long, MALIG, REACT, value="trim1_mean", min_donors=MIN_DONORS)
sel_wins = TS.paired_selectivity(long, MALIG, REACT, value="wins90_mean", min_donors=MIN_DONORS)
sel_clean = TS.paired_selectivity(long, MALIG, REACT, value="mean_all_clean",
                                  min_donors=MIN_DONORS)
sel_ds = TS.paired_selectivity(long, MALIG, REACT, value="mean_all_ds", min_donors=MIN_DONORS)
blk = TS.blocked_selectivity(long, MALIG, REACT, block=DONOR_STUDY, value="mean_all")
sel = sel.merge(blk, on="gene", how="left")

conc = TQ.concentration(long, MALIG)                       # top1/top5 share, detection
brn = TQ.burden_scaling(long, bur, MALIG, REACT)           # does the effect scale with burden
confine = (TQ.cluster_confinement(cluster_long, MALIG)
           if cluster_long is not None else None)

ev = TQ.assemble(sel, sel_trim, sel_clean, qc_cell, brn, conc, confine)
for name, s in (("wins", sel_wins), ("ds", sel_ds)):
    ev = ev.merge(s[["gene", "log2fc_paired"]].rename(
        columns={"log2fc_paired": f"log2fc_{name}"}), on="gene", how="left")
    ev[f"retained_{name}"] = TQ._ratio(ev[f"log2fc_{name}"], ev["log2fc_paired"])
ev = TQ.prevalence_residual(ev)

print(f"{len(sel):,} genes tested paired on {len(PAIRED_DONORS)} donors | "
      f"{int((sel['q'] < 0.05).sum()):,} at q < 0.05")

print("\n" + "=" * 78)
print("THE NULL: what pure contamination scores in this cohort")
print("=" * 78)
ok, msg = TQ.band_is_calibrated(ev, None)
print(msg)
band = TQ.contamination_band(ev)
print(f"\nlineage controls found: {band.attrs['n_genes']} "
      f"({', '.join(band.attrs['genes_found'])})")
print("\nper-gene, in full:")
_bcols = [c for c in ["gene", "log2fc_paired", "frac_donors_up", "median_pct_expr",
                      "retained_trim", "retained_clean", "retained_ds", "top1_share_excess",
                      *(f"{c}_excess" for c in TQ.coupling_cols()),
                      "or_topdecile_excess", "rho_burden"]
          if c in ev.columns]
print(ev[ev["gene"].isin(TQ.BAND_GENES)][_bcols]
      .sort_values("log2fc_paired", ascending=False)
      .to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("\nquantiles of the band (thresholds are read from the p90 row):")
print(band.round(4).to_string())

# The activity axis, which the band above cannot test. If the malignant arm is simply deeper,
# `retained_ds` -- the share of the effect that survives thinning every cell to a common depth --
# falls for the whole ranking at once, not for a handful of genes.
print("\n" + "=" * 78)
print("THE OTHER MECHANISM: how much of each effect survives depth matching")
print("=" * 78)
_up = ev[(ev["q"] < 0.05) & (ev["log2fc_paired"] > 0.3)]
print(f"genes up at q < 0.05 with log2FC > 0.3: {len(_up)}")
for nm, c in (("trim top 1%", "retained_trim"), ("winsorise p90", "retained_wins"),
              ("drop dirty cells", "retained_clean"), ("match depth", "retained_ds")):
    if c in _up:
        print(f"  {nm:18s} median retained {_up[c].median():.2f} | "
              f"{int((_up[c] < 0.5).sum())} of {len(_up)} genes keep less than half")
print("\nIf `match depth` alone collapses the ranking, the screen was measuring library size. "
      "If every column collapses together, it was measuring a handful of cells.")

In [ ]:
# ============================================================================
# §2b  The verdict -- which is only allowed to exclude anything if the null is calibrated
# ============================================================================
out = TQ.artifact_evidence(ev, band)
if not ok:
    # The lineage controls are the whole basis for calling a gene contamination. If they are not
    # up in the malignant arm, the mechanism is absent from this cohort and the six tests are
    # measuring something else. Nothing is excluded; the evidence is still printed, because the
    # columns are informative about every gene whether or not contamination is the explanation.
    out["artifact_flag"] = "clean"
    out["fail_reasons"] = "- (null not calibrated; no exclusion applied)"
    print("*" * 78)
    print("NO GENE IS EXCLUDED ON CONTAMINATION GROUNDS.")
    print(msg)
    print("*" * 78)

ev = out
print()
print(TQ.verdict_text(ev))

print("\n" + "=" * 78)
print("the six tests, per gene, for every gene with a positive effect at q < 0.05")
print("=" * 78)
_full = ev[(ev["q"] < 0.05) & (ev["log2fc_paired"] > 0)].sort_values("log2fc_paired",
                                                                    ascending=False)
_fc = [c for c in ["gene", "log2fc_paired", "frac_donors_up", "median_pct_expr",
                   "retained_trim", "retained_wins", "retained_clean", "retained_ds",
                   "top1_share_excess", *(f"{c}_excess" for c in TQ.coupling_cols()),
                   "r_umi_excess", "or_topdecile_excess", "rho_burden", "top_cluster_share",
                   "n_fails", "fail_reasons"] if c in ev.columns]
print(_full[_fc].to_string(index=False, float_format=lambda v: f"{v:.4g}"))

ev.to_csv(ST.tab("artifact_evidence"), index=False)
band.to_csv(ST.tab("contamination_band"))
print("\nwrote", ST.tab("artifact_evidence"), "and", ST.tab("contamination_band"))
print("\nColumn key:  retained_* = effect kept after trimming the top 1% of cells / "
      "winsorising at the 90th percentile / dropping lineage-positive and high-doublet cells / "
      "binomially thinning to a common depth. 1.0 = unchanged, 0 = gone. *_excess = the "
      "statistic minus the cohort median for genes detected at the same rate. rho_burden = "
      "Spearman of the per-donor effect against that donor's malignant:reactive ratio.")

In [ ]:
# ============================================================================
# §2c FX  The null, and where the disputed genes actually sit
# ============================================================================
SETS = {"lineage controls\n(pure contamination)": TQ.BAND_GENES,
        "all screened genes": sorted(set(ev["gene"]) - set(TQ.BAND_GENES)),
        "disputed genes": TQ.SUSPECTS,
        "validated antigens": ST.POSITIVE_CONTROLS}
fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.4),
                         gridspec_kw={"width_ratios": [1.05, 1.0]})

ax = axes[0]
rng = np.random.default_rng(0)
for i, (lab, gs) in enumerate(SETS.items()):
    v = ev.loc[ev["gene"].isin(gs), "log2fc_paired"].dropna()
    col = [TF.DOWN, "#bdbdbd", "#2e86c1", TF.UP][i]
    ax.scatter(np.full(len(v), i) + rng.normal(0, .07, len(v)), v, s=7 if i == 1 else 18,
               color=col, alpha=.35 if i == 1 else .9, linewidths=0)
    if len(v):
        ax.hlines(np.median(v), i - .25, i + .25, color="#111111", lw=1.4, zorder=5)
ax.axhline(0, lw=.5, c="k")
ax.set_xticks(range(len(SETS)))
ax.set_xticklabels(SETS, fontsize=6)
ax.set_ylabel("log2FC, malignant vs reactive CD4")
ax.set_title("Contamination null vs candidates", loc="left", fontsize=7.5)
TF._clean(ax)

# Right: the three-arm view. Contamination that rides into the malignant arm has no reason to
# spare healthy CD4 -- a healthy skin biopsy carries keratinocytes and fibroblasts too, and more
# of them than a dense infiltrate does. A transcript absent from healthy and reactive CD4 and
# present in the clone is tumour-restricted, whatever one thinks of its gene name.
ax = axes[1]
show = TQ.SUSPECTS + ST.POSITIVE_CONTROLS + ["CD7"]
det = (long[long["gene"].isin(show) & long["arm"].isin([HC_CD4, REACT, MALIG])]
       .pivot_table(index="gene", columns="arm", values="pct_expr", aggfunc="median")
       .reindex([g for g in show if g in set(long["gene"])])
       .sort_values(MALIG))
for j, (arm, col, mk) in enumerate(((HC_CD4, TF.COMPARATOR, "o"), (REACT, TF.REACT, "o"),
                                    (MALIG, TF.MALIG, "o"))):
    if arm in det:
        ax.scatter(det[arm], np.arange(len(det)) + (j - 1) * 0.2, s=22, color=col,
                   marker=mk, linewidths=0, label=arm.replace("CD4_", ""))
for y, (_, r_) in enumerate(det.iterrows()):
    ax.plot([r_.min(), r_.max()], [y, y], color="#dddddd", lw=0.8, zorder=0)
ax.set_yticks(range(len(det)))
ax.set_yticklabels([TF.it(g) for g in det.index], fontsize=6.5)
ax.set_xlabel("median fraction of a donor's cells detected")
ax.set_xscale("symlog", linthresh=1e-3)
ax.set_xlim(left=0)
ax.set_ylim(-0.7, len(det) - 0.3)
ax.legend(fontsize=5.5, frameon=False, loc="upper left", title="CD4 arm",
          title_fontsize=5.5)
ax.set_title("Detection across CD4 arms", loc="left", fontsize=7.5)
TF._clean(ax)

fig.tight_layout()
TF.save(fig, "FX_artifact_null", caption_text=(
    "Left: paired log2FC by gene set. Right: median detection per CD4 arm, symlog x."))
TF.save_source(det.reset_index(), "FX_three_arm_detection_source")
TF.logfig("FX_artifact_null", "contamination null vs candidates; three-arm detection",
          len(ev))

print("median detection by arm, the genes in question (and the controls):")
print(det.round(4).to_string())

In [ ]:
# ============================================================================
# §2d  Two more ways the effect could be an artifact, and neither is contamination
# ============================================================================
# (1) CLUSTER CIRCULARITY. The skin malignancy call is `mal_cluster` -- a *cluster* label. The
#     arms are therefore groups of Leiden clusters, and the comparison is between the same
#     clusters whose expression defined them. A gene confined to one or two malignant clusters
#     would be a cluster marker rediscovered as an antigen.
if confine is not None:
    print("share of the malignant arm's signal held by its single strongest cluster "
          f"({int(confine['n_clusters'].median())} clusters):")
    _q = confine["top_cluster_share"].describe(percentiles=[.5, .9, .99]).round(4)
    print("  all screened genes:", _q.to_dict())
    print("\n  the genes in question:")
    print(confine[confine["gene"].isin(TQ.SUSPECTS + ST.POSITIVE_CONTROLS)]
          .sort_values("top_cluster_share", ascending=False)
          .to_string(index=False, float_format=lambda v: f"{v:.4g}"))
    print("\n  A gene spread evenly over N clusters scores 1/N. Confinement would show as a "
          "share far above that floor.")

# (2) THE CALL ITSELF. `mal_tcr` assigns malignancy from the expanded TCR clone, which is
#     measured independently of the transcriptome. If an effect holds under that call it cannot
#     be an artifact of how the clusters were drawn. Built by:
#         bash jobs/run_target_screen_qc.sh --tissue skin --call mal_tcr --no-cluster \
#              --out-dir data/targets/by_call_mal_tcr
TCR_DIR = TARGETS / "by_call_mal_tcr"
_keep = ["gene", "log2fc_paired", "frac_donors_up", "n_donors_paired"]
if (TCR_DIR / f"{TISSUE}_target_long.parquet").exists():
    long_tcr = TC.filter_long(pd.read_parquet(TCR_DIR / f"{TISSUE}_target_long.parquet"), don)
    sel_tcr = TS.paired_selectivity(long_tcr, MALIG, REACT, value="mean_all",
                                    min_donors=MIN_DONORS)
    both_calls = sel[_keep].merge(sel_tcr[_keep], on="gene", suffixes=("_cluster", "_tcr"))
    _r = stats.spearmanr(both_calls["log2fc_paired_cluster"], both_calls["log2fc_paired_tcr"])
    print("\n" + "=" * 78)
    print(f"the same screen with the arms built from the TCR clone instead of the cluster: "
          f"rho = {_r.statistic:.2f} over {len(both_calls):,} genes "
          f"({both_calls['n_donors_paired_tcr'].median():.0f} paired donors vs "
          f"{both_calls['n_donors_paired_cluster'].median():.0f})")
    print("=" * 78)
    print(both_calls[both_calls["gene"].isin(TQ.SUSPECTS + ST.POSITIVE_CONTROLS
                                             + ST.LOSS_MARKERS)]
          .sort_values("log2fc_paired_cluster", ascending=False)
          .to_string(index=False, float_format=lambda v: f"{v:.3g}"))
    both_calls.to_csv(ST.tab("call_sensitivity"), index=False)
    print("\nAn effect that is an artifact of the clustering cannot survive a call made from "
          "the TCR. One that does survive is a property of the clone. Skin TCR recovery is "
          "patchier than blood's, so the paired-donor counts above are the limit on this "
          "control, not the correlation.")
    print("wrote", ST.tab("call_sensitivity"))
else:
    print(f"\n{TCR_DIR.name} not built -- the cluster-circularity control is MISSING. "
          "Run: bash jobs/run_target_screen_qc.sh --tissue skin --call mal_tcr --no-cluster "
          f"--out-dir {TCR_DIR}")

## §3 — Malignant vs reactive CD4, on what survived

The comparison the whole screen exists for, now on an independent cohort of MF/SS patients with a
CD4 clone, and on genes that passed §2. Paired, because both arms come from the same biopsy and
the same library prep: the donor is the block, and an unpaired test would spend its power on
between-patient variation that pairing removes. The study-blocked test (van Elteren) sits beside
it for the donors carrying only one arm, never instead of it.

Five figures, in the style of `28_tme_degradome_mc`, because a cohort median hides the thing that
decides whether a target is usable: **what it does inside each patient.**

| figure | what it shows |
|---|---|
| **FA** | the leading antigens, one grey line per donor joining its reactive and malignant means |
| **FB** | the antigens whose answer is already known — read this one first |
| **FC** | the ranked candidate list as a dumbbell, with prevalence and off-tumour beside each row |
| **FD** | the donor × gene effect matrix, donors ordered by malignant burden |
| **FE** | the same comparison on *detection rate*, which is what a CAR is dosed against |

The star above each gene is the BH-adjusted paired Wilcoxon; `n↑/n↓` underneath it is the donor
vote, and the two percentages under each gene name are the median share of cells detected in the
malignant and reactive arms. Titles say what the panel is; the finding is in the table printed
under it.

FD is the figure to argue with. Donors run left to right by malignant share of their CD4
compartment, so a row that darkens towards the right is a row whose effect tracks arm composition
— the pattern §2 tests for formally with `fails_burden`, shown here so it can also be seen. In
skin that ordering is close to a patch → plaque → tumour ordering, so a row that darkens to the
right may also be a real lesion-stage effect; 47 separates the two.

In [ ]:
# ============================================================================
# §3  Malignant vs reactive CD4, on the genes that survived §2
# ============================================================================
prev = TS.prevalence(long, MALIG, detect_floor=DETECT_FLOOR)
off = TS.off_tumour(long, target_arms=[MALIG], healthy_t_arms=[HC_CD4], value="pct_expr")
off = off.rename(columns={"worst_offtumour": "worst_T_arm_pct",
                          "worst_offtumour_arm": "worst_T_arm"})

core = (sel.merge(prev, on="gene", how="left")
           .merge(off, on="gene", how="left")
           .merge(ev[["gene", "artifact_flag", "n_fails", "fail_reasons",
                      "retained_trim", "retained_clean", "retained_ds", "rho_burden"]],
                  on="gene", how="left"))
core = core.merge(ST.annotate(core["gene"])[["gene", "cd_name", "panel_class", "stage"]],
                  on="gene", how="left")

GATES = ((core["artifact_flag"].eq("clean")) & (core["q"] < 0.05)
         & (core["log2fc_paired"] > 0.5) & (core["frac_donors_positive"] >= 0.5)
         & (~core["gene"].isin(ST.LOSS_MARKERS)) & (~core["gene"].isin(TQ.BAND_GENES)))
up = core[GATES].sort_values("log2fc_paired", ascending=False)
SHORT_GENES = up["gene"].tolist()[:30]
TOP_GENES = SHORT_GENES[:12]

CAPTION = (f"Paired Wilcoxon within donor, BH. {len(PAIRED_DONORS)} donors, "
           f"{meta['study'].nunique()} studies. log1p CP10K.")
LINE_CAPTION = CAPTION + " Lines: red = up in malignant, blue = down."

print(f"{len(up)} genes pass every gate "
      "(clean, q<0.05, log2FC>0.5, detected in >=50% of donors):\n")
_c = ["gene", "log2fc_paired", "frac_donors_up", "frac_donors_positive", "median_pct_expr",
      "q", "z_blocked", "retained_trim", "retained_clean", "retained_ds", "rho_burden",
      "worst_T_arm", "cd_name", "stage"]
print(up[[c for c in _c if c in up]].to_string(index=False,
                                               float_format=lambda v: f"{v:.4g}"))

print("\nthe loss phenotype -- CD7 and DPP4/CD26 loss is the textbook CTCL immunophenotype, and "
      "is the sanity check that the cleaning did not flatten real biology:")
print(core[core["gene"].isin(ST.LOSS_MARKERS)][["gene", "log2fc_paired", "frac_donors_up",
                                                "q", "artifact_flag"]]
      .sort_values("log2fc_paired").to_string(index=False,
                                              float_format=lambda v: f"{v:.4g}"))

print("\npaired log2FC vs the study-blocked z, over every tested gene -- the two tests should "
      "agree, and where they do not the gene is carried by one study:")
_ok = core.dropna(subset=["z_blocked"])
print(f"  r = {np.corrcoef(_ok['log2fc_paired'], _ok['z_blocked'])[0, 1]:.2f} "
      f"over {len(_ok):,} genes")

In [ ]:
# ============================================================================
# §3 FA  The leading surviving antigens, one line per donor
# ============================================================================
recA = TF.t_paired(long, TOP_GENES, MALIG, REACT, value="mean_all", min_donors=MIN_DONORS)
recA = recA.merge(ST.annotate(recA["gene"])[["gene", "cd_name", "stage"]], on="gene", how="left")

fig, ax = plt.subplots(figsize=(0.82 * len(recA) + 1.8, 3.6))
TF.paired_slopes(ax, recA)
ax.set_title(f"Top antigens, malignant vs reactive CD4 (n={len(PAIRED_DONORS)})",
             loc="left", fontsize=7.5)
TF.save(fig, "FA_top_candidates_paired", caption_text=LINE_CAPTION)
TF.save_source(recA, "FA_top_candidates_paired_source")
TF.logfig("FA_top_candidates_paired", "leading clean antigens, paired per donor", len(recA))

TF.show_tests(recA, cols=TF.SHOW_COLS + ["cd_name", "stage"])

In [ ]:
# ============================================================================
# §3 FB  Calibration: the antigens whose answer is already known
# ============================================================================
# Read this panel before FA. CCR4 (mogamulizumab, approved in CTCL), TNFRSF8/CD30 (brentuximab,
# approved), KIR3DL2 (lacutamab) and IL2RA are the positive controls; CD7 and DPP4/CD26 loss IS
# the CTCL immunophenotype, so they must come out strongly negative. The v1 skin screen failed
# exactly here -- CCR4 386th, IL2RA negative -- which is why this panel decides whether anything
# in FA is evidence. All four controls are testable in skin: the object is not an HVG subset.
CTRL_GENES = [g for g in (ST.POSITIVE_CONTROLS + ST.LOSS_MARKERS)
              if g in set(long["gene"])]
recB = TF.t_paired(long, CTRL_GENES, MALIG, REACT, value="mean_all", min_donors=MIN_DONORS)
recB = recB.merge(ev[["gene", "artifact_flag"]], on="gene", how="left")

fig, ax = plt.subplots(figsize=(1.0 * len(recB) + 1.6, 3.4))
TF.paired_slopes(ax, recB)
ax.set_title("Validated antigens and CTCL loss markers", loc="left", fontsize=7.5)
TF.save(fig, "FB_controls_paired", caption_text=LINE_CAPTION)
TF.save_source(recB, "FB_controls_paired_source")
TF.logfig("FB_controls_paired", "positive controls and loss markers, paired", len(recB))

TF.show_tests(recB, cols=TF.SHOW_COLS + ["artifact_flag"])
_missing = sorted(set(ST.POSITIVE_CONTROLS + ST.LOSS_MARKERS) - set(long["gene"]))
if _missing:
    print(f"\nnot testable, absent from the screened gene set: {_missing}")
print("\nCD26 is an alias of DPP4, which is present under that symbol. A positive control that "
      "sits in the middle of the ranking is a calibration failure and the arms are the first "
      "suspect -- not a discovery about that drug.")

In [ ]:
# ============================================================================
# §3 FC  The surviving candidates as one ranked panel
# ============================================================================
recC = TF.t_paired(long, SHORT_GENES, MALIG, REACT, value="mean_all", min_donors=MIN_DONORS)
recC = recC.merge(ST.annotate(recC["gene"])[["gene", "cd_name", "stage"]], on="gene", how="left")
recC = recC.merge(core[["gene", "frac_donors_positive", "median_pct_expr", "worst_T_arm",
                        "worst_T_arm_pct"]], on="gene", how="left")
recC = recC.merge(prev[["gene", "n_donors"]].rename(columns={"n_donors": "n_donors_malig"}),
                  on="gene", how="left")
recC["ylabel"] = [f"{TF.it(r['gene'])}" + (f" · {r['cd_name']}" if r["cd_name"] else "")
                  for _, r in recC.iterrows()]
recC = recC.sort_values("log2fc", ascending=False)

fig, _ = TF.shortlist_panel(recC, floor=DETECT_FLOOR,
                            title="Shortlist: effect, patients positive, off-tumour burden")
TF.save(fig, "FC_shortlist_dumbbell", caption_text=CAPTION)
TF.save_source(recC, "FC_shortlist_dumbbell_source")
TF.logfig("FC_shortlist_dumbbell", "ranked clean candidates, both arms", len(recC))

TF.show_tests(recC, cols=TF.SHOW_COLS + ["cd_name", "stage", "frac_donors_positive",
                                         "median_pct_expr", "worst_T_arm", "worst_T_arm_pct"])

In [ ]:
# ============================================================================
# §3 FD  Leading markers inside each sample: the donor x gene effect matrix
# ============================================================================
M = TH.lfc_matrix(long, MALIG, REACT, genes=TOP_GENES, min_donors=MIN_DONORS)
M = M.loc[[g for g in TOP_GENES if g in M.index]]
# donors ordered by their own malignant burden, so the eye can check the one thing §2 warned
# about: an effect that grows left to right is an effect that grows with arm composition.
_b = bur["burden"].reindex(M.columns).dropna().sort_values()
M = M[list(_b.index)]

fig, ax = plt.subplots(figsize=(0.13 * M.shape[1] + 3.2, 0.19 * len(M) + 1.8))
TF.donor_gene_heat(ax, M, stars_by=dict(zip(core["gene"], core["q"])))
ax.set_xlabel(f"donor, ordered by malignant share of CD4 "
              f"({_b.iloc[0]:.0%} → {_b.iloc[-1]:.0%})")
ax.set_title("Per-donor log2FC, malignant vs reactive", loc="left", fontsize=7.5)
TF.save(fig, "FD_donor_gene_heat", caption_text=CAPTION)
TF.save_source(M.reset_index(), "FD_donor_gene_heat_source")
TF.logfig("FD_donor_gene_heat", "per-donor log2FC matrix, donors by burden", M.size)

print("per-donor log2FC, complete matrix (genes x donors):")
print(M.round(3).to_string())
print("\nper-gene summary of that matrix:")
_sum = pd.DataFrame({
    "n_donors": M.notna().sum(1),
    "median": M.median(1), "iqr": M.quantile(.75, axis=1) - M.quantile(.25, axis=1),
    "min": M.min(1), "max": M.max(1),
    "frac_up": (M > 0).sum(1) / M.notna().sum(1),
    "frac_up_1": (M > 1).sum(1) / M.notna().sum(1),
}).sort_values("median", ascending=False)
print(_sum.to_string(float_format=lambda v: f"{v:.3g}"))
print("\n`frac_up_1` -- donors where the antigen is at least doubled -- is the column that "
      "separates a cohort-wide antigen from one that is strong in a subset. 47 takes that "
      "apart formally.")

In [ ]:
# ============================================================================
# §3 FE  Detection, not expression: the axis a CAR is actually dosed against
# ============================================================================
recE = TF.t_paired(long, TOP_GENES, MALIG, REACT, value="pct_expr", min_donors=MIN_DONORS)
fig, ax = plt.subplots(figsize=(10.2, 3.4))
TF.paired_slopes(ax, recE, ylabel="fraction of a donor's cells detected", show_detection=False)
ax.set_title("Fraction of CD4 cells detected, per donor", loc="left", fontsize=7.5)
TF.save(fig, "FE_detection_paired", caption_text=LINE_CAPTION)
TF.save_source(recE, "FE_detection_paired_source")
TF.logfig("FE_detection_paired", "paired detection rate per donor", len(recE))

print("Detection is the quantity that decides whether a CAR sees the tumour; expression is the "
      "quantity the ranking is sorted on. They are not the same ordering, and where they "
      "disagree the detection column wins for a cell-therapy target.\n")
TF.show_tests(recE)

print("\n" + "=" * 78)
print("eligibility: in what fraction of patients would at least X% of the tumour be positive")
print("=" * 78)
elig = TH.eligibility(long, MALIG, genes=TOP_GENES, donors=PAIRED_DONORS)
print(elig.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
elig.to_csv(ST.tab("eligibility_skin"), index=False)
print("\nwrote", ST.tab("eligibility_skin"))

## §4 — On-target off-tumour in skin

This is the axis where skin is **stronger than blood**, and it is worth saying why: 43 can only
report the T and NK arms, because the non-T blood compartments were never put through the screen
builder, so a myeloid or HSPC hit is invisible there. Here the whole tissue is available.

Three costs, never averaged:

1. **tissue toxicity** — the worst non-T compartment in MF skin (keratinocyte, fibroblast,
   endothelium, myeloid, mast …), from the degradome pseudobulk cube: sample-level summed counts
   over all 42,347 genes, which is enough to answer "is this also on keratinocytes".
2. **healthy skin** — Reynolds 2021, a different lab, chemistry, site and sex. It cannot be
   blocked at all, so it is used to flag a candidate, never to rank one.
3. **T-cell aplasia** — expression on healthy CD4 and on the reactive arm. For CTCL this is often
   accepted: CD7, CD5 and CD4 agents all cause it. It is a cost to state, not a filter to hide.

A skin-directed agent and a systemic one do not pay these in the same currency: a keratinocyte
hit is a blistering risk for an intralesional agent and a non-issue for an antibody that never
reaches the epidermis at dose. The columns stay separate for that reason.

In [ ]:
# ============================================================================
# §4a  Off-tumour: the other T arms, then the non-T skin compartments
# ============================================================================
print("worst non-target T arm, for the surviving candidates:")
_o = (core[core["gene"].isin(SHORT_GENES)]
      [["gene", "cd_name", "worst_T_arm", "worst_T_arm_pct", "healthy_t_burden",
        "median_pct_expr", "frac_donors_positive"]]
      .sort_values("worst_T_arm_pct", ascending=False))
print(_o.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\n`healthy_t_burden` is the aplasia cost: the share of HEALTHY CD4 cells that carry the "
      f"antigen, over {meta.loc[meta['arm'].eq(HC_CD4), 'donor'].nunique()} healthy-skin donors. "
      f"`worst_T_arm` ranges over {sorted(set(meta['arm']) - {MALIG})}.")

# non-T skin compartments, from the degradome cube (CP10K means, sample level)
cube = pd.read_parquet(dcfg.NB_DIR / "data" / "degradome" /
                       "skin_degradome_pseudobulk_full.parquet")
cmeta = pd.read_parquet(dcfg.NB_DIR / "data" / "degradome" /
                        "skin_degradome_pseudobulk_meta.parquet")
genes = [g for g in prev["gene"] if g in cube.columns]
lib = cube.sum(axis=1).to_numpy()[:, None]
cp10k = np.log1p(cube[genes].to_numpy() / np.where(lib == 0, 1, lib) * 1e4)
nonT = cmeta["degradome_level"].astype(str)
is_nonT = ~nonT.str.startswith(("CD4", "CD8", "Treg", "NK", "ILC", "Prolif"))
print(f"\nnon-T levels in the cube: {sorted(nonT[is_nonT].unique())}")

cells_ok = cmeta["n_cells"].to_numpy() >= MIN_CELLS
nt = pd.DataFrame(cp10k[is_nonT.to_numpy() & cells_ok], columns=genes)
nt["level"] = nonT[is_nonT.to_numpy() & cells_ok].to_numpy()
_med = nt.groupby("level").median(numeric_only=True)
worst_nonT = (_med.max().rename("worst_nonT_skin").reset_index()
              .rename(columns={"index": "gene"}))
worst_nonT_lvl = (_med.idxmax().rename("worst_nonT_level").reset_index()
                  .rename(columns={"index": "gene"}))
off = off.merge(worst_nonT, on="gene", how="left").merge(worst_nonT_lvl, on="gene", how="left")

print("\nthe surviving candidates against the worst non-T skin compartment:")
_r = (off[off["gene"].isin(SHORT_GENES)]
      .merge(prev[["gene", "median_mean_all"]], on="gene", how="left")
      .assign(nonT_over_tumour=lambda d: d["worst_nonT_skin"] - d["median_mean_all"])
      .sort_values("nonT_over_tumour", ascending=False))
print(_r[["gene", "worst_nonT_skin", "worst_nonT_level", "median_mean_all",
          "nonT_over_tumour"]].to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\n`nonT_over_tumour` > 0 means the gene is higher on some non-T skin cell than on the "
      "tumour arm itself. That is a tissue-toxicity flag for a skin-directed agent and a "
      "contamination flag for the screen -- the two readings are not exclusive, which is why "
      "§2 tests the second one per cell rather than inferring it from this column.")

print("\nthe worst offenders over ALL screened genes, for scale:")
_all = (off.merge(prev[["gene", "median_mean_all"]], on="gene", how="left")
        .sort_values("worst_nonT_skin", ascending=False).head(15))
print(_all[["gene", "worst_nonT_skin", "worst_nonT_level", "median_mean_all"]]
      .to_string(index=False, float_format=lambda v: f"{v:.3g}"))

In [ ]:
# ============================================================================
# §4b  Healthy skin (Reynolds 2021) -- different lab, chemistry, site and sex
# ============================================================================
# The external arm cannot be blocked at all, so a shift in a gene with no biological reason to
# differ is the platform floor, not a result. It is used to flag candidates that are high on
# healthy skin cells, never to rank them.
try:
    ext = X.load_external()
    ext_genes = [g for g in prev["gene"] if g in set(map(str, ext.var_names))]
    lvl = ext.obs[X.TARGET_COL].astype(str)
    Xe = ext[:, ext_genes].X
    Xe = np.asarray(Xe.todense()) if hasattr(Xe, "todense") else np.asarray(Xe)
    ext_med = (pd.DataFrame(Xe, columns=ext_genes).assign(level=lvl.to_numpy())
               .groupby("level").median(numeric_only=True))
    ext_worst = ext_med.max().rename("worst_healthy_skin").reset_index().rename(
        columns={"index": "gene"})
    ext_worst["worst_healthy_skin_level"] = ext_med.idxmax().to_numpy()
    off = off.merge(ext_worst, on="gene", how="left")
    print(f"healthy-skin arm: {ext.n_obs:,} cells, {lvl.nunique()} levels, "
          f"{len(ext_genes)} of the screened genes present")
    print("\nthe surviving candidates on healthy skin:")
    print(off[off["gene"].isin(SHORT_GENES)]
          [["gene", "worst_healthy_skin", "worst_healthy_skin_level", "worst_nonT_skin",
            "worst_nonT_level"]]
          .sort_values("worst_healthy_skin", ascending=False)
          .to_string(index=False, float_format=lambda v: f"{v:.3g}"))
    print("\nA candidate high on healthy skin is a candidate whose on-target toxicity does not "
          "need the disease to be present. Read it as a flag against the lesion-restricted "
          "story, not as a measured toxicity.")
except Exception as e:                      # the reference is optional, the screen is not
    print("healthy-skin arm unavailable:", e)
    print("-> off-tumour is atlas-internal only; say so wherever the table is quoted.")

## §5–§6 — Surface plausibility and escape risk

**§5, trafficking.** A transcript is the first step of a long pipeline: translocation, folding,
glycosylation, export, endocytosis, recycling, shedding. Two things are reported, both context
rather than correction — the **module scores per arm** (is the malignant arm's secretory and
recycling machinery in a different state from the reactive arm, which is when a
transcript-to-surface inference is least safe) and the **per-gene caveat** beside each candidate.
`SELL` is the worked example: its transcript says nothing without `ADAM17` beside it.

This is also where the §2 activity axis reappears as biology rather than as an artifact. If every
trafficking module is up in the malignant arm, that is the same "larger, busier cell" that
inflates the housekeeping ranking — and it means a transcript-to-surface inference is *least*
reliable exactly for the genes the screen ranks highest. 45 asks the sharper version of this
question against measured CITE-seq protein, which exists in blood only; **skin has no protein
layer at all**, so for skin this section is the end of the argument rather than a step in it.

**§6, escape risk.** An antigen whose detection rate swings between the per-sample clusters of
**one** patient is the one that selects an antigen-negative clone under therapy — the failure
mode of every single-antigen CAR in a heterogeneous tumour. The clusters are the same ones the
malignancy call was made on.

In [ ]:
# ============================================================================
# §5  Trafficking machinery, per arm
# ============================================================================
tr_genes = [g for g in ST.TRAFFIC_GENES if g in set(long["gene"])]
tr = long[long["gene"].isin(tr_genes)].copy()
tr["module"] = tr["gene"].map(lambda g: ST.GENE_TO_TRAFFIC.get(g, [""])[0])

# z across donors within a module, so a module is compared to itself, not to other modules
piv = tr.groupby(["arm", "donor", "module"])["mean_all"].mean().reset_index()
piv["z"] = piv.groupby("module")["mean_all"].transform(lambda s: (s - s.mean()) / (s.std() or 1))
arm_mod = piv.pivot_table(index="module", columns="arm", values="z")
keep = [c for c in (MALIG, REACT, HC_CD4) if c in arm_mod.columns]
arm_mod = arm_mod[keep]
print("trafficking module z by arm (mean over donors) -- the capacity to put a protein on the "
      "surface and keep it there:")
print(arm_mod.round(2).to_string())

if MALIG in arm_mod and REACT in arm_mod:
    delta = (arm_mod[MALIG] - arm_mod[REACT]).sort_values()
    print("\nmalignant - reactive, by module:")
    print(delta.round(2).to_string())
    print(f"\n{int((delta > 0).sum())} of {len(delta)} modules are up in the malignant arm.")
    print("A large positive delta on `shedding` or `endocytosis` means the arm that looks "
          "antigen-high by transcript may be the arm that keeps least of it on the surface. "
          "Every module up at once is the biosynthetic-activity axis of §2, and it is the "
          "reason a transcript ranking in this compartment should be read with the "
          "`retained_ds` column in hand.")

TRAFFIC_DELTA = (arm_mod[MALIG] - arm_mod[REACT]).to_dict() if (
    MALIG in arm_mod and REACT in arm_mod) else {}

In [ ]:
# ============================================================================
# §5b  Per-gene tier + the machinery each candidate must be read against
# ============================================================================
tier = pd.DataFrame({"gene": prev["gene"]})
tier["surface_tier"] = [
    "caveated" if g in ST.SURFACE_CAVEAT else "unknown" for g in tier["gene"]]
tier["surface_caveat"] = tier["gene"].map(ST.SURFACE_CAVEAT).fillna("")

# SELL is the worked example: its transcript says nothing without ADAM17 beside it.
pair_rows = []
for g, partner in [("SELL", "ADAM17"), ("TNFRSF8", "ADAM17"), ("IL2RA", "ADAM17"),
                   ("CTLA4", "AP2M1"), ("PDCD1", "FBXO38"), ("CCR4", "ARRB2")]:
    if g in set(long["gene"]) and partner in set(long["gene"]):
        a = long[long["gene"].eq(g)].groupby("arm")["mean_all"].median()
        b = long[long["gene"].eq(partner)].groupby("arm")["mean_all"].median()
        pair_rows.append({"gene": g, "partner": partner,
                          "gene_malig": a.get(MALIG, np.nan), "gene_react": a.get(REACT, np.nan),
                          "partner_malig": b.get(MALIG, np.nan),
                          "partner_react": b.get(REACT, np.nan)})
pairs = pd.DataFrame(pair_rows)
print("candidate beside the machine that removes it from the surface:")
print(pairs.round(3).to_string(index=False))
print("\nRead each row as a pair. A higher transcript with a higher sink is not a higher "
      "epitope.")

_cav = tier[tier["gene"].isin(SHORT_GENES) & tier["surface_caveat"].ne("")]
print("\ndocumented surface caveats among the surviving candidates:")
print(_cav[["gene", "surface_caveat"]].to_string(index=False) if len(_cav)
      else "  none of the surviving candidates has a documented internalisation, shedding or "
           "lattice mechanism in `surface_targets.SURFACE_CAVEAT` -- which means nobody has "
           "looked, not that there is none")

In [ ]:
# ============================================================================
# §6  Escape risk
# ============================================================================
esc = TS.escape_risk(long, MALIG, cluster_long=cluster_long)
print("escape risk computed on",
      "per-sample clusters" if cluster_long is not None else "donors only")
cols = [c for c in ("gene", "between_donor_sd", "within_donor_sd", "min_donor_pct") if c in esc]

print("\nthe surviving candidates, by within-patient heterogeneity:")
print(esc[esc["gene"].isin(SHORT_GENES)]
      .merge(prev[["gene", "median_pct_expr"]], on="gene", how="left")
      .sort_values("within_donor_sd" if "within_donor_sd" in esc else "between_donor_sd",
                   ascending=False)
      [cols + ["median_pct_expr"]]
      .to_string(index=False, float_format=lambda v: f"{v:.3g}"))

print("\n`between_donor_sd` is the spread of detection across patients and `within_donor_sd` "
      "the spread across one patient's own clusters -- a high second column is the "
      "antigen-negative subclone a single-target agent would select for, and `min_donor_pct` is "
      "the worst patient. 47 takes the between-patient term apart properly.")

hi = esc.merge(prev[["gene", "median_pct_expr", "frac_donors_positive"]], on="gene")
hi = hi[hi["frac_donors_positive"] >= 0.5].sort_values(
    "within_donor_sd" if "within_donor_sd" in hi else "between_donor_sd", ascending=False)
print("\nfor scale, over every prevalent gene -- most within-patient heterogeneity:")
print(hi.head(10)[cols + ["median_pct_expr"]].to_string(index=False,
                                                        float_format=lambda v: f"{v:.3g}"))
print("\nand the most uniform:")
print(hi.tail(10)[cols + ["median_pct_expr"]].to_string(index=False,
                                                        float_format=lambda v: f"{v:.3g}"))

## §7 — The table, the control check that licenses reading it, and the plausibility layer

No composite score and no cutoff: the axes stay in their own columns and the filtering decision
belongs next to the sentence that justifies the threshold.

**Run the control check first.** CCR4, TNFRSF8, KIR3DL2 and IL2RA are approved or registrational
in CTCL. If they are not near the top, the screen is miscalibrated and nothing below them is
evidence — that is what happened to the v1 skin pass, and it is the single number by which this
rebuild should be judged.

Then the layer the statistics cannot supply. `surface_confident`, `n_tm` and `surfy_class` come
from CSPA and Surfy, not from this cohort: a transcript can be reproducibly, tumour-restrictedly
up and still be a hopeless antibody or CAR target because the protein is cytoplasmic, secreted or
mitochondrial. In skin that is not a hypothetical — ENO1, VDAC1, GPI and CALU all passed the v1
statistics.

In [ ]:
# ============================================================================
# §7  Assemble, check the controls, shortlist
# ============================================================================
surf = ST.load_surfaceome(strict=False)
rank = TS.rank_table(prev, sel, off, esc, ST.annotate, surfaceome=surf)
rank = rank.merge(tier[["gene", "surface_tier", "surface_caveat"]], on="gene", how="left")
rank = rank.merge(ev[["gene", "artifact_flag", "n_fails", "fail_reasons", "retained_trim",
                      "retained_clean", "retained_ds", "rho_burden", "top1_share"]],
                  on="gene", how="left")
if "top_cluster_share" in ev:
    rank = rank.merge(ev[["gene", "top_cluster_share"]], on="gene", how="left")

hit, missing, n_tested = TS.control_check(rank, ST.POSITIVE_CONTROLS)
print(f"positive controls of {n_tested:,} ranked genes:")
print(hit.to_string(index=False))
if missing:
    print(f"*** not testable: {missing} ***")
_worst = hit["rank"].max() if len(hit) else np.nan
print(f"\nworst control rank: {_worst} of {n_tested:,} "
      f"({_worst / n_tested:.0%} of the list)" if len(hit) else "")
print("A control outside the top ~10% is a calibration failure, not a discovery. The v1 pass "
      "put the worst control at 1,968 of 2,115 -- compare that number with the one above "
      "before reading anything below.")

short = rank[rank["gene"].isin(SHORT_GENES)].copy()
show = ["gene", "cd_name", "frac_donors_positive", "median_pct_expr", "log2fc_paired", "q",
        "worst_T_arm", "healthy_t_burden", "worst_nonT_skin", "worst_nonT_level",
        "worst_healthy_skin", "between_donor_sd", "stage", "n_evidence", "surface_confident",
        "n_tm", "surfy_class", "surface_tier", "artifact_flag"]
print(f"\n{len(short)} candidates pass the gates:")
print(short[[c for c in show if c in short]]
      .to_string(index=False, float_format=lambda v: f"{v:.4g}"))

print("\n" + "=" * 78)
print("PLAUSIBILITY, which the statistics cannot decide")
print("=" * 78)
print("`surface_confident` / `n_tm` / `surfy_class` come from CSPA + Surfy, not from this "
      "cohort. Genes with no surfaceome evidence are the ones to argue about before anyone "
      "orders a reagent:")
_noev = short[short["surface_confident"].isna() | ~short["surface_confident"].astype(bool)]
print(_noev[["gene", "cd_name", "log2fc_paired", "median_pct_expr", "n_evidence",
             "surfy_class", "topology"]].to_string(index=False,
                                                   float_format=lambda v: f"{v:.3g}")
      if len(_noev) else "  every shortlisted gene has confident surfaceome evidence")

print("\nnovel = passes the gates AND has no entry in CLINICAL_PRECEDENT:")
novel = short[short["stage"].fillna("").eq("")]
print(novel[["gene", "cd_name", "frac_donors_positive", "log2fc_paired", "panel_class",
             "surface_confident", "surfy_class"]]
      .to_string(index=False, float_format=lambda v: f"{v:.3g}"))

rank.to_csv(ST.tab("candidates_skin"), index=False)
short.to_csv(ST.tab("shortlist_skin"), index=False)
print("\nwrote", ST.tab("candidates_skin"), rank.shape, "and", ST.tab("shortlist_skin"),
      short.shape)

In [ ]:
# ============================================================================
# §7b FG  Two views of the whole ranking: calibration, and the trade every target makes
# ============================================================================
d = rank.dropna(subset=["log2fc_paired", "frac_donors_positive"])
is_ctrl = d["gene"].isin(ST.POSITIVE_CONTROLS)
is_short = d["gene"].isin(short["gene"])
is_band = d["gene"].isin(TQ.BAND_GENES)

fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.4))

ax = axes[0]
ax.scatter(d["log2fc_paired"], d["frac_donors_positive"], s=4, c="#d5d8dc", linewidths=0)
ax.scatter(d.loc[is_band, "log2fc_paired"], d.loc[is_band, "frac_donors_positive"],
           s=12, facecolors="none", edgecolors=TF.DOWN, linewidths=.6,
           label="lineage controls")
ax.scatter(d.loc[is_short, "log2fc_paired"], d.loc[is_short, "frac_donors_positive"],
           s=10, c="#2e86c1", linewidths=0, label="shortlist")
ax.scatter(d.loc[is_ctrl, "log2fc_paired"], d.loc[is_ctrl, "frac_donors_positive"],
           s=26, c="#c0392b", linewidths=0, label="approved / registrational")
for _, r_ in d[is_ctrl].iterrows():
    ax.annotate(r_["gene"], (r_["log2fc_paired"], r_["frac_donors_positive"]), fontsize=6,
                xytext=(3, 3), textcoords="offset points")
ax.axvline(0, lw=0.5, c="k")
ax.set_xlabel("log2FC malignant vs same-donor reactive CD4")
ax.set_ylabel("fraction of donors positive")
ax.set_title("Selectivity against prevalence, whole ranking", loc="left", fontsize=7.5)
ax.legend(fontsize=5.5, frameon=False, loc="lower left")
TF._clean(ax)

ax = axes[1]
if "healthy_t_burden" in rank.columns:
    d2 = rank.dropna(subset=["log2fc_paired", "healthy_t_burden"])
    d2 = d2[d2["frac_donors_positive"] >= 0.3]
    k = d2["gene"].isin(ST.POSITIVE_CONTROLS) | d2["gene"].isin(SHORT_GENES[:12])
    ax.scatter(d2["healthy_t_burden"], d2["log2fc_paired"], s=5, c="#d5d8dc", linewidths=0)
    ax.scatter(d2.loc[k, "healthy_t_burden"], d2.loc[k, "log2fc_paired"], s=18, c="#c0392b",
               linewidths=0)
    for _, r_ in d2[k].iterrows():
        ax.annotate(r_["gene"], (r_["healthy_t_burden"], r_["log2fc_paired"]), fontsize=6,
                    xytext=(3, 2), textcoords="offset points")
    ax.axhline(0, lw=0.5, c="k")
    ax.set_xlabel("share of healthy CD4 cells positive  (T-cell aplasia cost)")
    ax.set_ylabel("log2FC malignant vs reactive")
    ax.set_title("Selectivity against healthy-T burden", loc="left", fontsize=7.5)
    TF._clean(ax)
else:
    ax.set_axis_off()
    ax.text(0.5, 0.5, "no CD4_HC arm in this run", ha="center", fontsize=7)

fig.tight_layout()
TF.save(fig, "FG_ranking_overview", caption_text=(
    f"{len(PAIRED_DONORS)} paired donors. Left: every ranked gene; red = approved or "
    "registrational in CTCL, open red = the contamination null. Right: donors positive >= 30%."))
TF.save_source(d[["gene", "log2fc_paired", "frac_donors_positive", "healthy_t_burden"]],
               "FG_ranking_overview_source")
TF.logfig("FG_ranking_overview", "selectivity vs prevalence and vs aplasia cost", len(d))

print("Where the validated antigens sit in the left panel is the calibration statement; where "
      "the lineage controls sit is the contamination statement. Both are read off one figure "
      "on purpose.")

## §8 — Dossier

What was screened, on which cohort and call, with which gates, and what the result is licensed to
say. The compartment comparison — which antigens separate tumour from reactive in **both** skin
and blood — lives in `43 §8`, which reads `tgt42_candidates_skin.csv`; with this rebuild both
sides of that comparison have been through the same cohort fix and the same artifact tests, so
the caveat 43 carried about an uncleaned skin table no longer applies.

In [ ]:
# ============================================================================
# §8  Dossier
# ============================================================================
def SEC(t): print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)
def kv(k, v): print(f"  {k:36s} {v}")

SEC("1 - INPUT")
kv("object", Path(man["source"]).name)
kv("builder", man.get("builder"))
kv("cells", f"{man['n_cells']:,}")
kv("genes in object", f"{man['n_genes_in_object']:,}")
kv("genes screened", f"{man['gene_selection'].get('n_total', '?'):,}")
kv("malignancy call", man["malignancy_call"])
kv("library for CP10K", man.get("library"))

SEC("2 - COHORT")
kv("donor keys in the object", len(don))
for reason, n in don.loc[~don["cohort_keep"], "drop_reason"].value_counts().items():
    kv(f"dropped: {reason}", n)
kv("independent keys", int(don["cohort_keep"].sum()))
kv("people", don.loc[don["cohort_keep"], "person"].nunique())
kv("donors with both CD4 arms", len(PAIRED_DONORS))
print()
print(pd.crosstab(don.loc[don["cohort_keep"], "dx"],
                  don.loc[don["cohort_keep"], "study"]).to_string())

SEC("3 - ARMS"); print(arms.to_string())

SEC("4 - GATES")
kv("min cells per donor-arm", MIN_CELLS)
kv("min paired donors", MIN_DONORS)
kv("detection floor", DETECT_FLOOR)
kv("shortlist", "clean, q<0.05, paired log2FC>0.5, >=50% donors positive, "
                "not a loss marker, not a lineage control")

SEC("5 - ARTIFACT TESTS")
kv("lineage controls found", band.attrs["n_genes"])
kv("coupling modules", TQ.COUPLING_MODULES[TISSUE])
kv("null calibrated", ok)
print("  " + msg)
print()
print(ev["artifact_flag"].value_counts().to_string())
kv("\n  genes excluded from the headline",
   int((ev["artifact_flag"].ne("clean") & (ev["log2fc_paired"] > 0)).sum()))

SEC("6 - CALIBRATION"); print(hit.to_string(index=False))
if missing:
    print(f"  not testable: {missing}")

SEC("7 - WHAT THIS IS NOT")
for line in [
    "A target call. It is a transcript ranking.",
    "A measure of epitope density, accessibility, turnover or shedding. Skin carries no ADT, "
    "so the protein check 45 runs for blood has no equivalent here.",
    "Blocked for the healthy-skin arm: Reynolds 2021 is a different lab, chemistry, site and "
    "sex, and that arm flags a candidate rather than ranking it.",
    "Able to see genes the screened set does not carry -- absent is not unexpressed.",
    "Independent of the malignancy call: every arm inherits mal_cluster's errors, and §2d's "
    "TCR rebuild is the only control for that.",
    "A statement about patients whose clone is not CD4, who are now out of the cohort "
    "entirely, or about LyP and parapsoriasis, which were never this screen's question.",
]:
    print("  - " + line)

SEC("8 - OUTPUTS")
TF.write_index("figure_index")
for p in sorted(ST.TAB_DIR.glob(f"{PREFIX}*")) + sorted(ST.FIG_DIR.glob(f"{PREFIX}*")):
    print("  ", p.relative_to(NB_DIR))

In [ ]:
# ============================================================================
# §9  The complete numerical result, as one block to hand to another model
# ============================================================================
# Everything this notebook computed, with a sentence on what each number is and how it was
# produced. Printed here and written to tables/tgt42_llm_export.md; paste the whole block
# between the banner lines into any other LLM and it needs no other file.
_ctx = f"""
Compartment: MF/CTCL **skin**, {man['n_cells']:,} T/NK cells from
{Path(man['source']).name} ({man['n_genes_in_object']:,} genes in the object; the screen tests
the {man['gene_selection'].get('n_total', 0):,}-gene surface panel union surfaceome selection
plus lineage and trafficking controls).

Cohort: {len(don)} donor keys reduced to {int(don['cohort_keep'].sum())} independent keys
({don.loc[don['cohort_keep'], 'person'].nunique()} people) by dropping non-CD4 clones,
diagnoses outside the MF/SS spectrum, and repeat biopsies of one patient. Diagnosis from the
curated `clin_dx`. {len(PAIRED_DONORS)} donors carry both a malignant and a reactive CD4 arm and
enter the paired test.

Arms: malignancy call `{man['malignancy_call']}`; CD4 cells are split into CD4_malignant,
CD4_reactive, CD4_unassessed (never used as a comparator) and CD4_HC (healthy donors), beside
CD4_Treg and CD8.

Headline test: per gene, the within-donor difference of mean log1p CP10K between the malignant
and reactive arms, paired Wilcoxon over donors, BH-adjusted (`q`). `log2fc_paired` is the median
per-donor log2 ratio; `frac_donors_up` the share of donors with a positive difference.
`z_blocked` is the study-blocked van Elteren statistic, which also uses donors with one arm.

Known answers, for calibration: CCR4 (mogamulizumab) and TNFRSF8/CD30 (brentuximab) are approved
in CTCL, KIR3DL2 is lacutamab's target, IL2RA is registrational. CD7 and DPP4/CD26 loss is the
textbook CTCL immunophenotype and must come out negative. The previous version of this screen
failed that check (worst control 1,968 of 2,115), which is what prompted the rebuild.
"""

_sections = [
    {"name": "§0b Cohort", "what": "which donor keys are independent MF/SS patients with a CD4 "
                                   "clone, and why each dropped key was dropped.",
     "how": "curated clin_* columns only (docs/CLINICAL_METADATA.md); no expression used.",
     "table": don[don["cohort_keep"]][["study", "dx", "dx2", "lesion", "lesion_class",
                                       "clin_stage", "clin_stage_class", "sex", "age",
                                       "is_cultured", "n_cells", "burden"]],
     "file": str(ST.tab("skin_cohort").relative_to(NB_DIR)), "max_rows": 120},
    {"name": "§0b Dropped keys", "what": "the keys removed from the cohort, with the reason.",
     "how": "same source; the three rules are non-CD4 clone, off-spectrum diagnosis, repeat "
            "biopsy of a patient already in the cohort.",
     "table": don.loc[~don["cohort_keep"], ["study", "dx", "phenotype", "lesion", "n_cells",
                                            "drop_reason"]]},
    {"name": "§1 Arms", "what": "donors, cells and studies per arm after the cohort fix.",
     "how": "groups of >= 20 cells.", "table": arms},
    {"name": "§1 Contamination load per arm",
     "what": "how dirty each arm is: share of lineage-positive cells, library size, doublet "
             "score and the tissue module scores.",
     "how": "median over the arm's donors. A malignant arm dirtier than its reactive arm is the "
            "precondition for the contamination story in §2.",
     "table": meta.groupby("arm")[_cl].median()},
    {"name": "§2a Contamination null",
     "what": "what pure contamination scores here: the six statistics computed on transcripts a "
             "CD4 T cell cannot make (keratins, collagens, VWF, PMEL, tryptases, LYZ, HBB).",
     "how": "same paired test and per-cell couplings as every other gene. Thresholds for the "
            "artifact flags are read off the p90 row of this band, not chosen.",
     "table": ev[ev["gene"].isin(TQ.BAND_GENES)][_bcols]},
    {"name": "§2a Calibration verdict",
     "what": "whether the contamination mechanism is operating in this cohort at all.",
     "how": "the null is calibrated only if the lineage controls are themselves up in the "
            "malignant arm (median log2FC > 0.1 and median frac_donors_up > 0.5). If not, no "
            "gene is excluded on contamination grounds.",
     "table": {"null_calibrated": ok, "verdict": msg}},
    {"name": "§2b Artifact verdict, per gene",
     "what": "the six tests and the resulting flag for every gene up at q < 0.05.",
     "how": "fails_trim / fails_clean / fails_ds are the share of the effect retained after "
            "trimming the top 1% of cells, dropping lineage-positive and high-doublet cells, "
            "and depth-matching; fails_concentration / fails_lineage / fails_umi are "
            "prevalence-matched residuals against the band; fails_burden is the Spearman of the "
            "per-donor effect against that donor's malignant:reactive ratio. Two failures = "
            "artifact, one = suspect.",
     "table": _full[_fc], "file": str(ST.tab("artifact_evidence").relative_to(NB_DIR)),
     "max_rows": 80},
    {"name": "§3 Candidates passing every gate",
     "what": "the genes that are selectively up on malignant CD4 and survived §2.",
     "how": "clean, q < 0.05, paired log2FC > 0.5, detected in >= 50% of donors, not a loss "
            "marker, not a lineage control.",
     "table": up[[c for c in _c if c in up]]},
    {"name": "§3 Loss markers",
     "what": "the CTCL loss phenotype, as the sanity check that cleaning did not flatten real "
             "biology.", "how": "same paired test.",
     "table": core[core["gene"].isin(ST.LOSS_MARKERS)][["gene", "log2fc_paired",
                                                        "frac_donors_up", "q",
                                                        "artifact_flag"]]},
    {"name": "§3 FD Per-gene summary of the donor x gene effect matrix",
     "what": "how consistent each leading antigen's effect is across patients.",
     "how": "per-donor log2FC matrix; frac_up_1 is the share of donors where the antigen is at "
            "least doubled.", "table": _sum},
    {"name": "§3 FE Eligibility",
     "what": "in what fraction of patients at least X% of the malignant CD4 cells carry the "
             "antigen.", "how": "transcript detection per donor, thresholded.",
     "table": elig, "file": str(ST.tab("eligibility_skin").relative_to(NB_DIR))},
    {"name": "§4 Off-tumour",
     "what": "what else carries each candidate: the worst non-target T arm, healthy CD4, the "
             "worst non-T skin compartment and healthy skin.",
     "how": "T arms from this screen's long table (share of cells detected); non-T compartments "
            "from the degradome pseudobulk cube (log1p CP10K, sample level, median per level); "
            "healthy skin from Reynolds 2021, which is unblocked and flags rather than ranks.",
     "table": short[[c for c in ("gene", "worst_T_arm", "worst_T_arm_pct", "healthy_t_burden",
                                 "worst_nonT_skin", "worst_nonT_level", "worst_healthy_skin",
                                 "worst_healthy_skin_level") if c in short]]},
    {"name": "§5 Trafficking modules per arm",
     "what": "the state of the machinery that puts a protein on the surface and keeps it there.",
     "how": "per-module z across donors, then the arm mean. Context, not a correction factor.",
     "table": arm_mod},
    {"name": "§6 Escape risk",
     "what": "how much each candidate's detection varies between patients and between one "
             "patient's own clusters.",
     "how": "SD of detection across donors and, within a donor, across its per-sample clusters.",
     "table": esc[esc["gene"].isin(SHORT_GENES)][cols]},
    {"name": "§7 Control check",
     "what": "where the antigens with a known answer sit in the ranking. This decides whether "
             "anything else here is evidence.",
     "how": "rank by paired log2FC over all screened genes.", "table": hit},
    {"name": "§7 Shortlist with the plausibility layer",
     "what": "the final candidate table: effect, prevalence, off-tumour cost, heterogeneity, "
             "clinical precedent and surfaceome evidence.",
     "how": "surface_confident / n_tm / surfy_class come from CSPA and Surfy, not from this "
            "cohort; they are the check that a statistically clean hit is a protein an antibody "
            "could reach.",
     "table": short[[c for c in show if c in short]],
     "file": str(ST.tab("shortlist_skin").relative_to(NB_DIR))},
]
TF.llm_export("42", "Surface-target screen, skin", _ctx, _sections)